# IMPORTS

In [1]:
import os
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle

# CONFIGURAÇÕES

In [2]:
BASE_FOLDER_TRAIN = "treino"

TRAIN_FILE = "ep2-train.csv"

preprocess_params = {
    "lowercase": False,
    "normalize_unicode": False,
    "remove_extra_whitespace": False,
    "remove_punct": False,
}

In [3]:
# Configuração dos Pipelines e Grades de Hiperparâmetros para Grid Search
param_grids = {
    'Logistic Regression': {
        'pipeline': Pipeline([
            ('vectorizer', TfidfVectorizer()),
            ('model', LogisticRegression(max_iter=1000, random_state=42))
        ]),
        'params': {
            'vectorizer__max_features': [10000],
            'vectorizer__ngram_range': [(1, 2)],
            'model__C': [1.0],
            'model__solver': ['lbfgs'],
            'model__class_weight': ['balanced', None]
        }
    }
}


# ANÁLISE DE BALANCEAMENTO DOS DATASETS


In [ ]:
def analisar_balanceamento(file_name):
    """Função para analisar o balanceamento de um dataset"""
    path = os.path.join(BASE_FOLDER_TRAIN, file_name)
    df = pd.read_csv(path, sep=";", encoding="latin1")
    
    print("="*60)
    print(f"ANÁLISE ESTATÍSTICA - {file_name}")
    print("="*60)
    
    # Informações básicas
    print(f"\n📊 INFORMAÇÕES GERAIS:")
    print(f"   • Total de linhas: {len(df):,}")
    print(f"   • Total de colunas: {len(df.columns)}")
    print(f"   • Colunas: {list(df.columns)}")
    
    # Verificar valores nulos
    print(f"\n🔍 VALORES NULOS:")
    print(f"   • Coluna 'req_text': {df['req_text'].isna().sum()}")
    print(f"   • Coluna 'profession': {df['profession'].isna().sum()}")
    
    # Distribuição das classes
    print(f"\n📈 DISTRIBUIÇÃO DAS CLASSES:")
    contagem_classes = df['profession'].value_counts()
    print(contagem_classes)
    
    print(f"\n📊 PORCENTAGEM POR CLASSE:")
    porcentagem_classes = df['profession'].value_counts(normalize=True) * 100
    for classe, perc in porcentagem_classes.items():
        count = contagem_classes[classe]
        print(f"   • {classe}: {count:,} ({perc:.2f}%)")
    
    # Verificar balanceamento
    print(f"\n⚖️ BALANCEAMENTO:")
    razao = contagem_classes.max() / contagem_classes.min()
    print(f"   • Razão maior/menor classe: {razao:.2f}x")
    if razao < 1.5:
        print(f"   • Status: ✅ Dataset bem balanceado")
    elif razao < 3:
        print(f"   • Status: ⚠️ Dataset moderadamente desbalanceado")
    else:
        print(f"   • Status: ❌ Dataset desbalanceado")
    
    print("\n" + "="*60)
    print()
    
    return df, contagem_classes


In [5]:
resultados_analise = {}

df, contagem = analisar_balanceamento(TRAIN_FILE)
resultados_analise[TRAIN_FILE] = {
    'dataframe': df,
    'contagem_classes': contagem
}


ANÁLISE ESTATÍSTICA - ep2-train.csv

📊 INFORMAÇÕES GERAIS:
   • Total de linhas: 43,678
   • Total de colunas: 2
   • Colunas: ['req_text', 'profession']

🔍 VALORES NULOS:
   • Coluna 'req_text': 0
   • Coluna 'profession': 0

📈 DISTRIBUIÇÃO DAS CLASSES:
profession
government    18782
academic      14593
private       10303
Name: count, dtype: int64

📊 PORCENTAGEM POR CLASSE:
   • government: 18,782 (43.00%)
   • academic: 14,593 (33.41%)
   • private: 10,303 (23.59%)

⚖️ BALANCEAMENTO:
   • Razão maior/menor classe: 1.82x
   • Status: ⚠️ Dataset moderadamente desbalanceado




# PRÉ-PROCESSAMENTO

In [8]:
def preprocess_operations(text, params):
    if not isinstance(text, str):
        return ""
    if params.get("normalize_unicode", True):
        text = unicodedata.normalize("NFKC", text)
    if params.get("lowercase", True):
        text = text.lower()
    if params.get("remove_punct", True):
        text = re.sub(r"[^\w\s]", " ", text)
    if params.get("remove_extra_whitespace", True):
        text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_data(path):
    if not os.path.exists(path):
        print(f"Aviso: {path} não encontrado.")
        return None

    df = pd.read_csv(path, sep=";", encoding="latin1")
    col_text, col_label = "req_text", "profession"

    df = df[[col_text, col_label]].dropna()
    df = shuffle(df, random_state=10).reset_index(drop=True)

    df["text_preproc"] = df[col_text].apply(lambda x: preprocess_operations(x, preprocess_params))

    le = LabelEncoder()
    y = le.fit_transform(df[col_label])
    X = df["text_preproc"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, stratify=y, random_state=10
    )

    return X_train, X_test, y_train, y_test

In [9]:
dataset = {}

path = os.path.join(BASE_FOLDER_TRAIN, TRAIN_FILE)
print(f"\nProcessando: {TRAIN_FILE}")
result = preprocess_data(path) 

if result is not None:
    X_train, X_test, y_train, y_test = result
    dataset = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }


Processando: ep2-train.csv


# TREINAMENTO

Classificação de textos entre autores **acadêmicos**, **privados** e **governamentais**.

In [10]:
X_train = dataset["X_train"]
X_test = dataset["X_test"]
y_train = dataset["y_train"]
y_test = dataset["y_test"]

print(f"Dados carregados")
print(f"   Treino: {len(X_train)} textos | Teste: {len(X_test)} textos")


Dados carregados
   Treino: 37126 textos | Teste: 6552 textos


In [12]:
print("="*80)
print("GRID SEARCH COM PIPELINE")
print("="*80)
print("Otimizando TF-IDF + Modelos simultaneamente...\n")

# Armazenar melhores pipelines
best_models = {}
cv_results = {}

for name, config in param_grids.items():
    print(f"[{name}] Executando Grid Search...")
    print(f"   Testando {len(config['params']['vectorizer__max_features']) * len(config['params']['vectorizer__ngram_range'])} combinações de TF-IDF...")
    
    # Grid Search com 10-fold CV
    grid_search = GridSearchCV(
        config['pipeline'],
        config['params'],
        cv=10,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X_train, y_train)
    
    # Armazenar resultados
    best_models[name] = grid_search.best_estimator_
    
    # Separar parâmetros de TF-IDF e modelo
    vectorizer_params = {k.replace('vectorizer__', ''): v 
                        for k, v in grid_search.best_params_.items() 
                        if k.startswith('vectorizer__')}
    model_params = {k.replace('model__', ''): v 
                   for k, v in grid_search.best_params_.items() 
                   if k.startswith('model__')}
    
    cv_results[name] = {
        'best_params': grid_search.best_params_,
        'vectorizer_params': vectorizer_params,
        'model_params': model_params,
        'best_score': grid_search.best_score_,
        'mean': grid_search.best_score_,
        'std': grid_search.cv_results_['std_test_score'][grid_search.best_index_]
    }
    
    print(f"  ✓ Melhores params TF-IDF: {vectorizer_params}")
    print(f"  ✓ Melhores params Modelo: {model_params}")
    print(f"  ✓ Acuracia (CV): {grid_search.best_score_:.4f}\n")

print("="*80)


GRID SEARCH COM PIPELINE
Otimizando TF-IDF + Modelos simultaneamente...

[Logistic Regression] Executando Grid Search...
   Testando 1 combinações de TF-IDF...
  ✓ Melhores params TF-IDF: {'max_features': 10000, 'ngram_range': (1, 2)}
  ✓ Melhores params Modelo: {'C': 1.0, 'class_weight': None, 'solver': 'lbfgs'}
  ✓ Acuracia (CV): 0.6615

